## 1. Base de dados textuais

**Justificativa e explicação da escolha:** Escolhemos o Project Gutenberg por reunir varias obras de domínio público. Cada livro vem acompanhada das seguintes caracteristicas: Autor, idioma, generos e etc..., o que permite usarmos os metadados para as tarefas de PLN.

**Adequação dos dados às tarefas de PLN (volume, abrangência, variedade):** 143 obras têm metadados colatados a partir da bookshelft 645, filtrando para o inglês e deduplicadas por título+autor (`work_key`). Com isso, baixamos 20 IDs, extraímos e limpamos 39 arquivos HTML e, após o cruzamento com os metadados, chegamos a 21 registros com texto completo e gênero rotulado.

**Limitação:** toda a base vem de uma única bookshelf, o que reduz a abrangência de gêneros/fontes fora desse recorte; e 21 exemplos é um volume pequeno para treinar um classificador robusto.

**Organização e interpretabilidade da base de dados (dicionário de dados):**

| Coluna | Tipo | Descrição |
|---|---|---|
| `book_id` | texto | Identificador único do livro no Project Gutenberg |
| `title` | texto | Título da obra |
| `author` | texto | Autor, no formato `Sobrenome, Nome, ano-ano` |
| `language` | texto | Idioma original da obra |
| `loc_class` | texto | Classificação da Library of Congress |
| `subjects` | texto | Assuntos associados à obra, separados por `\|` |
| `release_date` | data | Data de publicação no Gutenberg |
| `summary` | texto | Resumo/sinopse extraído da página do livro |
| `work_key` | texto | Chave normalizada (título+autor) usada para deduplicação |
| `file_path` | texto | Caminho do arquivo HTML baixado localmente |
| `text` | texto | Texto integral do livro, já limpo (sem HTML/licença) |
| `genre` | texto | Rótulo de gênero literário usado no classificador |

## 2. Script Python — scraping, acesso via API, outros métodos

**Funcionamento e reprodutibilidade:**O script funciona corretamente, contando com a coleta de metadados, downloada de texto integral, limpeza, rotulagem e treino de classificador. É parcialmente reprodutívelÇ o download verifica arquivos já existentes antes de baixar de novo, mas a paginação dos metadados exige um ajuste manual.

**Pontos de atenção identificados:** o User-Agent usado no download de texto integral imita um navegador comum, diferente do User-Agent identificável (com contato) usado na coleta de metadado, o ideal é manter identificação honesta em todas as requisições.

**Documentação dos procedimentos realizados:** documentado via células markdown explicando cada etapa (download, limpeza, construção do dataset, rotulagem, vetorização, classificação, teste), além de comentários no código.

**Criatividade / capacidade de resolução de problemas:** deduplicação de obras por título+autor normalizado (`work_key`) e extração de trechos contínuos de texto evitando cabeçalhos curtos.

In [1]:
import os
import time
import requests
import pandas as pd
import urllib3

# Silencia os avisos de SSL no terminal/notebook
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

output_folder = "gutenberg_harvest"
os.makedirs(output_folder, exist_ok=True)

# 1. Carrega os IDs do CSV para saber EXATAMENTE o que baixar
df_meta_source = pd.read_csv("gutenberg_books.csv", dtype={"book_id": str})
target_ids = df_meta_source["book_id"].dropna().str.strip().tolist()

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

MAX_DOWNLOADS = 20  # Quantidade de livros para baixar do CSV
downloaded = 0

print(f"Iniciando download baseado nos IDs do CSV (Meta: {MAX_DOWNLOADS} livros)...\n")

for book_id in target_ids:
    if downloaded >= MAX_DOWNLOADS:
        break

    # Verifica se o arquivo HTML ou ZIP já existe localmente
    already_downloaded = any(
        f.startswith(f"{book_id}-h") or f.startswith(f"pg{book_id}")
        for f in os.listdir(output_folder)
    )
    if already_downloaded:
        downloaded += 1
        print(f"[{downloaded}/{MAX_DOWNLOADS}] Livro ID {book_id} já existe na pasta. Pulando...")
        continue

    # Tenta as URLs padrões do Project Gutenberg para o ID
    urls_to_try = [
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-h.zip",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-h.htm",
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}-images.html"
    ]

    success = False
    for url in urls_to_try:
        try:
            r = requests.get(url, headers=headers, timeout=12, verify=False)
            if r.status_code == 200 and len(r.content) > 3000:
                ext = ".zip" if url.endswith(".zip") else ".htm"
                save_path = os.path.join(output_folder, f"{book_id}-h{ext}")
                with open(save_path, 'wb') as f:
                    f.write(r.content)

                downloaded += 1
                print(f"[{downloaded}/{MAX_DOWNLOADS}] Baixado ID {book_id} com sucesso!")
                success = True
                time.sleep(1)
                break
        except Exception:
            continue

    if not success:
        print(f"Não foi possível baixar o ID {book_id} pelas URLs padrão.")

print(f"\nDownload concluído! {downloaded} livros sincronizados com o CSV.")

Iniciando download baseado nos IDs do CSV (Meta: 20 livros)...

[1/20] Baixado ID 1342 com sucesso!
[2/20] Baixado ID 2701 com sucesso!
[3/20] Baixado ID 2554 com sucesso!
[4/20] Baixado ID 84 com sucesso!
[5/20] Baixado ID 11 com sucesso!
[6/20] Baixado ID 345 com sucesso!
[7/20] Baixado ID 43 com sucesso!
[8/20] Baixado ID 2641 com sucesso!
[9/20] Baixado ID 145 com sucesso!
[10/20] Baixado ID 65238 com sucesso!
[11/20] Baixado ID 3268 com sucesso!
[12/20] Baixado ID 67979 com sucesso!
[13/20] Baixado ID 37106 com sucesso!
[14/20] Baixado ID 1260 com sucesso!
[15/20] Baixado ID 2465 com sucesso!
[16/20] Baixado ID 59828 com sucesso!
[17/20] Baixado ID 2868 com sucesso!
[18/20] Baixado ID 244 com sucesso!
[19/20] Baixado ID 45839 com sucesso!
[20/20] Baixado ID 62215 com sucesso!

Download concluído! 20 livros sincronizados com o CSV.


## 3. Limpeza e preparação dos dados

**Dados foram tokenizados?** Sim, foi usado o nltk.tokenize.word_tokenize diretamente sobre o texto dos livros, para gerar uma coluna "tokens".

**Dados foram normalizados?** Sim, todo o texto para minúsculas por padrão antes de tokenizar. Além disso, o extract_text_from_html já colapsa espaços e quebras de linha redundantes.

**Remoção de ruídos:** Sim, BeautifulSoup remove tags do html, e uma expressão regular remove o cabeçalho/rodapé de licença do Project Gutenberg antes de qualquer análise do texto.

**Remoção de stopwords e pontuação:** Após a tokenização, stopwords e sinais de pontuação são filtrados dos tokens, gerando assim, a coluna "tokens_clean"

In [2]:
import os
import re
import glob
from bs4 import BeautifulSoup
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

nltk.download('stopwords')
nltk.download('punkt')

def extract_text_from_html(file_path):
    """
    Realiza o scraping do arquivo HTML baixado do livro,
    removendo tags HTML, cabeçalhos/rodapés e extraindo o texto limpo do conteúdo.
    """
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    # Remove scripts, estilos e cabeçalhos
    for script in soup(["script", "style", "header", "footer"]):
        script.extract()

    # Extrai texto do corpo da página
    text = soup.get_text(separator=" ")

    # Limpeza de marcas padrão do Project Gutenberg (Header e Footer da licença)
    start_match = re.search(r"\*\*\*\s*START OF TH(IS|E) PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.IGNORECASE)
    end_match = re.search(r"\*\*\*\s*END OF TH(IS|E) PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.IGNORECASE)

    if start_match and end_match:
        text = text[start_match.end():end_match.start()]
    elif start_match:
        text = text[start_match.end():]

    # Normalização básica do texto
    text = re.sub(r"\s+", " ", text).strip()
    return text

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [3]:
import zipfile

def build_dataset_from_harvest(base_folder):
    """
    Descompacta arquivos .zip da pasta harvest e varre os arquivos HTML
    para construir o DataFrame de livros.
    """
    # 1. Extrai todos os arquivos .zip baixados para a pasta local
    zip_files = glob.glob(os.path.join(base_folder, "*.zip"))
    print(f"Descompactando {len(zip_files)} arquivos .zip...")
    for zip_path in zip_files:
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(base_folder)
        except Exception as e:
            print(f"Erro ao descompactar {zip_path}: {e}")

    # 2. Varre todos os HTMLs gerados na descompactação
    html_files = glob.glob(os.path.join(base_folder, "**/*.htm*"), recursive=True)

    data = []
    print(f"Total de arquivos HTML encontrados: {len(html_files)}")

    for file_path in html_files:
        filename = os.path.basename(file_path)
        # Extrai ID do livro a partir do nome do arquivo (ex: 1342-h.htm -> 1342)
        match = re.search(r"(\d+)", filename)
        if not match:
            continue

        book_id = match.group(1)

        # Faz a raspagem do conteúdo textual
        cleaned_text = extract_text_from_html(file_path)

        # Filtra textos muito curtos ou vazios
        if len(cleaned_text) < 1000:
            continue

        data.append({
            "book_id": book_id,
            "file_path": file_path,
            "text": cleaned_text
        })

    df = pd.DataFrame(data)
    return df

df_books = build_dataset_from_harvest("gutenberg_harvest")
print(f"Livros válidos processados: {len(df_books)}")
df_books.head()

Descompactando 2 arquivos .zip...
Total de arquivos HTML encontrados: 20
Livros válidos processados: 20


,book_id,file_path,text
0,84,gutenberg_harvest/84-h.htm,Frankenstein | Project Gutenberg Frankenstein;...
1,67979,gutenberg_harvest/67979-h.htm,The Blue Castle | Project Gutenberg The BLUE C...
2,2701,gutenberg_harvest/2701-h.htm,Moby Dick; or The Whale | Project Gutenberg MO...
3,11,gutenberg_harvest/11-h.htm,Alice’s Adventures in Wonderland | Project Gut...
4,244,gutenberg_harvest/244-h.htm,A Study in Scarlet | Project Gutenberg A STUDY...


In [4]:
import nltk
import string

for pkg in ["punkt", "punkt_tab", "stopwords"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Aviso: não foi possível baixar {pkg}: {e}")

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

STOPWORDS_EN = set(stopwords.words("english"))
PUNCT = set(string.punctuation)


def tokenize_text(text):
    """Normaliza (minúsculas) e tokeniza o texto usando o NLTK."""
    return word_tokenize(text.lower(), language="english")


def remove_stopwords_punct(tokens):
    """Remove stopwords e pontuação da lista de tokens."""
    return [
        tok for tok in tokens
        if tok not in STOPWORDS_EN
        and not all(ch in PUNCT for ch in tok)
    ]


df_books["tokens"] = df_books["text"].apply(tokenize_text)
df_books["tokens_clean"] = df_books["tokens"].apply(remove_stopwords_punct)

df_books[["book_id"]].assign(
    n_tokens=df_books["tokens"].apply(len),
    n_tokens_clean=df_books["tokens_clean"].apply(len),
)

,book_id,n_tokens,n_tokens_clean
0,84,85329,35593
1,67979,83373,39338
2,2701,255797,115230
3,11,34648,15293
4,244,51592,22162
5,43,30662,12988
6,37106,235873,99119
7,345,189189,76891
8,1342,151395,63193
9,2868,107231,49850


## 4. Bonus round

**Stemming e lematização:** não implementado nesta versão.

**Bases adicionais (comparação entre fontes/recortes):** não implementado nesta versão.

**Automatização na coleta:** parcialmente implementado, o script de download verifica arquivos já existentes e evita baixá-los de novo, mas não há agendamento para atualização periódica automática.